In [40]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [41]:
import pandas as pd
import numpy as np
import torch
from transformers import pipeline
from sentence_transformers import CrossEncoder
import wandb
from kaggle_secrets import UserSecretsClient




In [42]:
device = 0 if torch.cuda.is_available() else -1
print("GPU:", torch.cuda.is_available())

def map_at_3(correct, predictions):
    for k, pred in enumerate(predictions[:3]):
        if pred == correct:
            return 1.0 / (k + 1)
    return 0.0

GPU: True


In [43]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

In [44]:
# Map@3 utility
def map_at_3(correct, predictions):
    for k, pred in enumerate(predictions[:3]):
        if pred == correct:
            return 1.0 / (k + 1)
    return 0.0

In [45]:
# DeBERTa Zero-Shot 

print("Loading DeBERTa...")
deberta = pipeline(
    "zero-shot-classification",
    model="cross-encoder/nli-deberta-v3-small",
    device=device
)

def get_deberta_scores(row):
    options = {opt: str(row[opt]) for opt in ['A','B','C','D','E']}
    result = deberta(
        str(row['prompt']),
        candidate_labels=list(options.values()),
        hypothesis_template="The correct answer to the question is: {}"
    )
    label_to_opt = {v: k for k, v in options.items()}
    # Return dict of {A: score, B: score, ...}
    return {label_to_opt[label]: score 
            for label, score in zip(result['labels'], result['scores'])}

Loading DeBERTa...


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-small
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [46]:
# facebook/bart-large-mnli   

print("Loading BART MNLI...")
bart = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device
)

def get_bart_scores(row):
    options = {opt: str(row[opt]) for opt in ['A','B','C','D','E']}
    result = bart(
        str(row['prompt']),
        candidate_labels=list(options.values()),
        hypothesis_template="The correct answer is: {}"
    )
    label_to_opt = {v: k for k, v in options.items()}
    return {label_to_opt[label]: score
            for label, score in zip(result['labels'], result['scores'])}

Loading BART MNLI...


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [47]:
# Precompute both models on 200 sample rows

sample = train.sample(200, random_state=42).reset_index(drop=True)

deberta_cache = []
bart_cache    = []
true_answers  = []

for i, (_, row) in enumerate(sample.iterrows()):
    deberta_cache.append(get_deberta_scores(row))
    bart_cache.append(get_bart_scores(row))
    true_answers.append(row['answer'])
    if i % 50 == 0:
        print(f"  {i}/200 done...")

print("Done!")

  0/200 done...
  50/200 done...
  100/200 done...
  150/200 done...
Done!


In [48]:
# Find best weights

def normalize(d):
    vals = list(d.values())
    mn, mx = min(vals), max(vals)
    if mx == mn:
        return {k: 1.0 for k in d}
    return {k: (v - mn) / (mx - mn) for k, v in d.items()}

weight_configs = [
    (1.0, 0.0),   # DeBERTa only
    (0.0, 1.0),   # BART only
    (0.5, 0.5),   # equal
    (0.6, 0.4),
    (0.4, 0.6),
    (0.7, 0.3),
    (0.3, 0.7),
]

best_map = 0
best_weights = None

print("Testing weight combinations...")
for w1, w2 in weight_configs:
    scores = []
    for s1, s2, ans in zip(deberta_cache, bart_cache, true_answers):
        n1, n2 = normalize(s1), normalize(s2)
        final = {opt: w1*n1[opt] + w2*n2[opt] for opt in ['A','B','C','D','E']}
        pred = ' '.join(sorted(final, key=final.get, reverse=True)[:3])
        scores.append(map_at_3(ans, pred.split()))
    avg = round(sum(scores)/len(scores), 4)
    marker = " ← best!" if avg > best_map else ""
    print(f"  DeBERTa={w1} BART={w2} → MAP@3: {avg}{marker}")
    if avg > best_map:
        best_map = avg
        best_weights = (w1, w2)

print(f"\n Best: DeBERTa={best_weights[0]}, BART={best_weights[1]} → {best_map}")

Testing weight combinations...
  DeBERTa=1.0 BART=0.0 → MAP@3: 0.5425 ← best!
  DeBERTa=0.0 BART=1.0 → MAP@3: 0.5858 ← best!
  DeBERTa=0.5 BART=0.5 → MAP@3: 0.5783
  DeBERTa=0.6 BART=0.4 → MAP@3: 0.57
  DeBERTa=0.4 BART=0.6 → MAP@3: 0.5842
  DeBERTa=0.7 BART=0.3 → MAP@3: 0.5417
  DeBERTa=0.3 BART=0.7 → MAP@3: 0.5883 ← best!

 Best: DeBERTa=0.3, BART=0.7 → 0.5883


In [49]:
# Full train validation

w1, w2 = best_weights

print("Validating on full train set...")
scores = []
for i, (_, row) in enumerate(train.iterrows()):
    s1 = get_deberta_scores(row)
    s2 = get_bart_scores(row)
    n1, n2 = normalize(s1), normalize(s2)
    final = {opt: w1*n1[opt] + w2*n2[opt] for opt in ['A','B','C','D','E']}
    pred = ' '.join(sorted(final, key=final.get, reverse=True)[:3])
    scores.append(map_at_3(row['answer'], pred.split()))
    if i % 200 == 0:
        print(f"  [{i}/2000] MAP@3: {round(sum(scores)/len(scores), 4)}")

full_map = round(sum(scores)/len(scores), 4)
print(f"\n Ensemble Train MAP@3: {full_map}")

Validating on full train set...
  [0/2000] MAP@3: 0.5
  [200/2000] MAP@3: 0.5995
  [400/2000] MAP@3: 0.579
  [600/2000] MAP@3: 0.5921
  [800/2000] MAP@3: 0.5847
  [1000/2000] MAP@3: 0.5819
  [1200/2000] MAP@3: 0.5799
  [1400/2000] MAP@3: 0.5784
  [1600/2000] MAP@3: 0.5757
  [1800/2000] MAP@3: 0.5783

 Ensemble Train MAP@3: 0.5814


In [50]:
# Generate submission 
predictions = []
for i, (_, row) in enumerate(test.iterrows()):
    s1 = get_deberta_scores(row)
    s2 = get_bart_scores(row)
    n1, n2 = normalize(s1), normalize(s2)
    final = {opt: w1*n1[opt] + w2*n2[opt] for opt in ['A','B','C','D','E']}
    pred = ' '.join(sorted(final, key=final.get, reverse=True)[:3])
    predictions.append({'id': row['id'], 'prediction': pred})
    if i % 50 == 0:
        print(f"  {i}/{len(test)} done...")
submission = pd.DataFrame(predictions)
submission.to_csv('submission.csv', index=False)
print(" submission.csv saved!")

  0/500 done...
  50/500 done...
  100/500 done...
  150/500 done...
  200/500 done...
  250/500 done...
  300/500 done...
  350/500 done...
  400/500 done...
  450/500 done...
 submission.csv saved!


In [ ]:
'''
This submission also uses an ensemble but now the models are changed from before.

This time, These models were combined to make the predictions:

* DeBERTa Zero-Shot Classification
* BART Zero-Shot Classification

Now here , 
For each question, both models assign a confidence score to all five answer options. 
Since the scores produced by the two models are different, 
they are first normalized and then combined using different weight combinations.

Several weight combinations were tested on a sample of the training dataset to identify the best-performing combination. 
The selected weights were then used to rank the answer options for every question in the test dataset. 
Finally, the top three ranked options were selected as the final prediction.

By using this method, the score obtained was 0.50623.

The Conclusion : This approach achieved a better score than previous submissions. 
The improvement shows that combining the predictions of two different Zero-Shot Classification models produces better results than using a single model. 
Since both models learn different patterns from the data, their combined predictions are more reliable.
Although the improvement is not very large, 
it indicates that ensemble methods do improve prediction accuracy but it also depends which models are chosen and their weights to combine the predictions. 
'''